[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brianjalaian/CAP6606_ML_ISR/blob/main/notebooks/01_random_forests_tabular.ipynb)

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/brianjalaian/CAP6606_ML_ISR/blob/main/notebooks/01_random_forests_tabular.ipynb)

# Random Forests and Tabular Data

This chapter covers decision tree ensembles and their application to structured tabular data—one of the most practical and widely-used approaches in machine learning.

## Learning Objectives

- Understand how decision trees work and their limitations
- Learn how random forests improve upon single decision trees
- Apply ensemble methods to tabular/structured data
- Interpret model predictions and feature importance
- Compare tree-based methods with neural network approaches

## Background

Random forests, introduced by Leo Breiman in 2001, revolutionized machine learning by providing a reliable algorithm that:

- Requires minimal assumptions about data form
- Needs little preprocessing
- Is fast to train and robust to overfitting
- Provides built-in feature importance

For tabular data (spreadsheets, databases, structured records), ensemble tree methods often outperform deep learning while being easier to interpret.

**References:**
- [Fast.ai Lesson 6: Random Forests](https://course.fast.ai/Lessons/lesson6.html)
- [Fastbook Chapter 9: Tabular Modeling](https://github.com/fastai/fastbook/blob/master/09_tabular.ipynb)
- Raschka, S. et al. *Machine Learning with PyTorch and Scikit-Learn* (2022), Chapters 3 & 7

## 1. Decision Trees

A decision tree recursively splits data based on feature thresholds to make predictions. Each split attempts to separate the data into more homogeneous groups.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
X.head()

In [ ]:
# Split data
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training samples: {len(X_train)}, Validation samples: {len(X_valid)}")

In [ ]:
# Train a single decision tree
tree = DecisionTreeRegressor(max_depth=5, random_state=42)
tree.fit(X_train, y_train)

train_pred = tree.predict(X_train)
valid_pred = tree.predict(X_valid)

print(f"Single Decision Tree (max_depth=5):")
print(f"  Train R²: {r2_score(y_train, train_pred):.4f}")
print(f"  Valid R²: {r2_score(y_valid, valid_pred):.4f}")

## 2. Random Forests

A random forest combines many decision trees trained on:
1. **Bootstrap samples** (random subsets of training data with replacement)
2. **Random feature subsets** at each split

This reduces overfitting and improves generalization through ensemble averaging.

In [ ]:
# Train a random forest
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

train_pred = rf.predict(X_train)
valid_pred = rf.predict(X_valid)

print(f"Random Forest (100 trees, max_depth=10):")
print(f"  Train R²: {r2_score(y_train, train_pred):.4f}")
print(f"  Valid R²: {r2_score(y_valid, valid_pred):.4f}")
print(f"  Valid RMSE: {np.sqrt(mean_squared_error(y_valid, valid_pred)):.4f}")

## 3. Feature Importance

Random forests provide built-in feature importance scores based on how much each feature contributes to reducing prediction error across all trees.

In [ ]:
# Extract and display feature importances
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance Ranking:")
print(importance_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Handling Categorical Variables

Tabular data often contains categorical features (e.g., city names, product types). These must be encoded numerically. Common approaches:

- **Label encoding**: Assign integers to categories (works well for trees)
- **One-hot encoding**: Create binary columns for each category
- **Embeddings**: Learn continuous representations (for neural networks)

In [ ]:
# Example with categorical data
from sklearn.preprocessing import LabelEncoder

# Create sample data with categorical feature
sample_df = pd.DataFrame({
    'region': ['North', 'South', 'East', 'West', 'North', 'South'],
    'size': [100, 150, 200, 120, 180, 90],
    'price': [250, 300, 400, 280, 350, 220]
})

# Label encode the categorical column
le = LabelEncoder()
sample_df['region_encoded'] = le.fit_transform(sample_df['region'])

print("Original and encoded data:")
print(sample_df)

## 5. Hyperparameter Tuning

Key hyperparameters for random forests:

| Parameter | Description | Typical Range |
|-----------|-------------|---------------|
| `n_estimators` | Number of trees | 100-1000 |
| `max_depth` | Maximum tree depth | 5-30 or None |
| `min_samples_leaf` | Minimum samples per leaf | 1-20 |
| `max_features` | Features considered per split | 'sqrt', 'log2', or fraction |

In [ ]:
# Quick comparison of different configurations
configs = [
    {'n_estimators': 50, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 200, 'max_depth': 15},
]

print("Hyperparameter Comparison:")
print("-" * 50)

for config in configs:
    model = RandomForestRegressor(**config, min_samples_leaf=5, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    score = r2_score(y_valid, model.predict(X_valid))
    print(f"n_estimators={config['n_estimators']:3d}, max_depth={config['max_depth']:2d} -> Valid R²: {score:.4f}")

## 6. Gradient Boosting: An Alternative Ensemble

While random forests train trees in parallel, **gradient boosting** trains trees sequentially, with each tree correcting errors from previous trees. Popular implementations include XGBoost, LightGBM, and CatBoost.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gb.fit(X_train, y_train)

valid_pred = gb.predict(X_valid)
print(f"Gradient Boosting:")
print(f"  Valid R²: {r2_score(y_valid, valid_pred):.4f}")
print(f"  Valid RMSE: {np.sqrt(mean_squared_error(y_valid, valid_pred)):.4f}")

## 7. When to Use Tree-Based Methods vs Neural Networks

**Tree-based methods (Random Forest, XGBoost):**
- Structured/tabular data with mixed feature types
- When interpretability matters
- Smaller datasets (< 100K rows)
- When you need fast iteration

**Neural networks:**
- Unstructured data (images, text, audio)
- Very large datasets
- When you can leverage transfer learning
- High-cardinality categorical features (use embeddings)

## Exercises

1. **Experiment with tree depth**: Train random forests with `max_depth` values from 3 to 20. Plot training vs validation R² to visualize overfitting.

2. **Out-of-bag score**: Random forests have a built-in validation mechanism. Set `oob_score=True` and compare `rf.oob_score_` with the validation score.

3. **Try XGBoost**: Install xgboost (`pip install xgboost`) and compare its performance with sklearn's GradientBoostingRegressor.

4. **Feature engineering**: Create new features (e.g., `rooms_per_household = AveRooms / AveOccup`) and measure the impact on model performance.

## Next Steps

- Review the lecture slides for theoretical foundations
- Explore the [fastai tabular tutorial](https://docs.fast.ai/tutorial.tabular.html) for neural network approaches
- Work through Chapter 7 of *Machine Learning with PyTorch and Scikit-Learn* for ensemble methods in depth